# 🎯 2. LLMs as Reasoning Engines

In notebook 01 we defined an AI Agent as `LLM + Tools + Memory + Loop`. The **LLM** is the brain — the reasoning engine that drives every decision.

In this notebook we explore:

1. **LLM fundamentals** — how autoregressive generation works
2. **Chat models and messages** — system, user, assistant roles
3. **Prompting as control** — system prompts define agent identity
4. **Structured output** — Pydantic models for reliable JSON
5. **Multi-model setup** — OpenAI, Anthropic, and beyond
6. **LLM limitations** — what the brain *cannot* do

In [1]:
import os
from pathlib import Path
from dotenv import load_dotenv

# Φορτώνουμε τα API keys από το .env (βρίσκεται στο root του project)
_env_path = Path(".env")
load_dotenv(dotenv_path=_env_path, override=False)

# Αν δεν βρεθεί το key (πχ σε Colab), ζητάμε manually
# if not os.environ.get("OPENAI_API_KEY"):
#     import getpass
#     os.environ["OPENAI_API_KEY"] = getpass.getpass("OpenAI API key: ")

LLM_MODEL   = "gpt-4o-mini"

print(f'Model: {LLM_MODEL}')
print(f'API key configured: {bool(os.getenv("OPENAI_API_KEY"))}')

Model: gpt-4o-mini
API key configured: True


## 2.1 How LLMs Generate Text

Large Language Models are **autoregressive**: they predict one token at a time, using all previous tokens as context.

<img src="images/llms-gen-text.png" width="80%" style="border-radius:10px;margin:12px 0;"/>

### Key Parameters

| Parameter | Effect | Agent Implication |
|-----------|--------|-------------------|
| **temperature** | Higher = more creative | Agents usually need `0.0-0.3` for reliable tool calls |
| **max_tokens** | Output length limit | Must budget tokens for reasoning + tool calls |
| **top_p** | Nucleus sampling | Alternative to temperature |
| **stop** | Stop sequences | Critical for agent loops (stop before hallucinating tool results) |

For agents, low temperature is generally preferred — we want **reliable, deterministic** decisions, not creative prose.

## 2.2 Chat Models and Message Roles

Modern LLMs use a **message-based** API with three roles:

| Role | Purpose | Agent Usage |
|------|---------|-------------|
| **system** | Define identity and rules | Agent persona, tool instructions |
| **user** | Human input | User queries, task descriptions |
| **assistant** | Model responses | Agent reasoning, tool call decisions |
| **tool** | Tool results | Observations fed back to the agent |

The **system message** is where agent behavior is defined:

In [2]:
from openai import OpenAI

client = OpenAI()

# The system message defines the agent's identity
messages = [
    {
        'role': 'system',
        'content': (
            'You are a senior Python developer and code reviewer. '
            'You analyze code for bugs, security issues, and performance problems. '
            'Always provide specific line references and severity ratings (LOW/MEDIUM/HIGH/CRITICAL).'
        )
    },
    {
        'role': 'user',
        'content': 'Review this code:\n\ndef login(user, pwd):\n    query = f"SELECT * FROM users WHERE name=\'{user}\' AND pass=\'{pwd}\'"\n    return db.execute(query)'
    }
]

response = client.chat.completions.create(
    model=LLM_MODEL,
    messages=messages,
    temperature=0.2
)

print(response.choices[0].message.content)

### Code Review

#### Issues Identified:

1. **SQL Injection Vulnerability**:
   - **Severity**: CRITICAL
   - **Line Reference**: Line 2
   - **Description**: The code constructs an SQL query using string interpolation, which makes it vulnerable to SQL injection attacks. An attacker could manipulate the `user` or `pwd` parameters to execute arbitrary SQL commands.

   **Recommendation**: Use parameterized queries or prepared statements to safely handle user input. For example, if using a library like `sqlite3`, the code should be modified as follows:
   ```python
   query = "SELECT * FROM users WHERE name=? AND pass=?"
   return db.execute(query, (user, pwd))
   ```

2. **Storing Passwords in Plain Text**:
   - **Severity**: HIGH
   - **Line Reference**: Line 2
   - **Description**: The code appears to be checking passwords directly against a database field. If the passwords are stored in plain text, this poses a significant security risk.

   **Recommendation**: Always hash passwords

## 2.3 Multi-Turn Conversations — Building Context

An agent's conversation history is its **working memory**. Each turn adds to the context that the LLM uses for reasoning.

In [3]:
# Multi-turn conversation demonstrating context accumulation

conversation = [
    {'role': 'system', 'content': 'You are a helpful AI assistant. Be concise.'},
    {'role': 'user', 'content': 'My name is Alex and I work at a startup.'},
]

# Turn 1
r1 = client.chat.completions.create(model=LLM_MODEL, messages=conversation)
assistant_msg = r1.choices[0].message.content
print(f'Assistant: {assistant_msg}\n')

# Add assistant response and new user message
conversation.append({'role': 'assistant', 'content': assistant_msg})
conversation.append({'role': 'user', 'content': 'What is my name and where do I work?'})

# Turn 2 — the model remembers!
r2 = client.chat.completions.create(model=LLM_MODEL, messages=conversation)
print(f'Assistant: {r2.choices[0].message.content}')
print(f'\nTotal messages in context: {len(conversation)}')

Assistant: Nice to meet you, Alex! How can I assist you today?

Assistant: Your name is Alex, and you work at a startup.

Total messages in context: 4


## 2.4 Structured Output with Pydantic

For agents, we often need the LLM to produce **structured data** — not free text. This is critical for:
- Parsing tool calls reliably
- Extracting structured information
- Making decisions that downstream code can process

OpenAI's `response_format` and LangChain's `with_structured_output()` make this reliable.

In [10]:
from pydantic import BaseModel, Field
from typing import List, Literal

# Define a structured output schema
class AgentDecision(BaseModel):
    """The agent's decision about what to do next."""
    thought: str = Field(description='The agent reasoning about the task')
    action: Literal['search', 'calculate', 'respond'] = Field(
        description='The action to take'
    )
    action_input: str = Field(description='Input for the chosen action')
    confidence: float = Field(ge=0.0, le=1.0, description='Confidence in the decision')

# Use structured output
response = client.beta.chat.completions.parse(
    model=LLM_MODEL,
    messages=[
        {'role': 'system', 'content': 'Decide what action to take for the user query.'},
        {'role': 'user', 'content': 'What is the square root of 144?'}
    ],
    response_format=AgentDecision
)

decision = response.choices[0].message.parsed
print(f'Thought:     {decision.thought}')
print(f'Action:      {decision.action}')
print(f'Input:       {decision.action_input}')
print(f'Confidence:  {decision.confidence}')
print(f'\nType: {type(decision)}')

Thought:     The user is asking for the square root of 144, which is a straightforward calculation. Since I can calculate mathematical operations directly, I will proceed with that.
Action:      calculate
Input:       sqrt(144)
Confidence:  0.9

Type: <class '__main__.AgentDecision'>


## 2.5 LangChain LLM Integration

LangChain provides a **unified interface** for working with multiple LLM providers. This is what we'll use throughout the rest of the course.

In [5]:
from langchain_openai import ChatOpenAI
from langchain_core.messages import HumanMessage, SystemMessage
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import ChatPromptTemplate

# Initialize the model
llm = ChatOpenAI(model=LLM_MODEL, temperature=0)

# Method 1: Direct invocation with message objects
response = llm.invoke([
    SystemMessage(content='You are a helpful assistant. Answer in one sentence.'),
    HumanMessage(content='What makes LLMs suitable as agent brains?')
])
print(f'Direct: {response.content}\n')

# Method 2: Using LCEL chains (the modern LangChain way)
chain = (
    ChatPromptTemplate.from_template('Explain {concept} in exactly two sentences.')
    | llm
    | StrOutputParser()
)

result = chain.invoke({'concept': 'function calling in AI agents'})
print(f'Chain: {result}')

Direct: LLMs are suitable as agent brains due to their ability to understand and generate human-like text, process complex information, and adapt to various contexts, enabling them to perform tasks that require reasoning, conversation, and decision-making.

Chain: Function calling in AI agents refers to the process where an agent invokes specific functions or methods to perform tasks or retrieve information based on its programming and the context of the interaction. This capability allows AI agents to execute predefined operations, manipulate data, and respond dynamically to user inputs or environmental changes.


## 2.6 Structured Output with LangChain

LangChain makes structured output even cleaner with `with_structured_output()`:

In [6]:
from pydantic import BaseModel, Field
from typing import List

class TaskAnalysis(BaseModel):
    """Analysis of whether a task needs an AI agent."""
    task_description: str = Field(description='Brief description of the task')
    needs_agent: bool = Field(description='Whether this task needs an agent')
    reasoning: str = Field(description='Why or why not an agent is needed')
    required_tools: List[str] = Field(description='Tools needed if agent is required')
    complexity: int = Field(ge=1, le=5, description='Complexity from 1-5')

# Create a structured LLM
structured_llm = llm.with_structured_output(TaskAnalysis)

# Analyze different tasks
tasks = [
    'Translate this paragraph to French',
    'Research the top 5 competitors and create a comparison report',
    'Monitor stock prices and alert me when AAPL drops below $150'
]

for task in tasks:
    analysis = structured_llm.invoke(f'Analyze this task: {task}')
    print(f'Task: {task}')
    print(f'  Needs Agent: {analysis.needs_agent}')
    print(f'  Complexity:  {"★" * analysis.complexity}{"☆" * (5 - analysis.complexity)}')
    print(f'  Tools:       {analysis.required_tools}')
    print(f'  Reasoning:   {analysis.reasoning}\n')

Task: Translate this paragraph to French
  Needs Agent: False
  Complexity:  ★★☆☆☆
  Tools:       []
  Reasoning:   The task of translating a paragraph to French can be performed by a human or a simple translation tool. It does not require advanced AI capabilities, as it is a straightforward language translation task that can be handled by existing translation software or a bilingual individual.

Task: Research the top 5 competitors and create a comparison report
  Needs Agent: True
  Complexity:  ★★★★☆
  Tools:       ['Web scraping tools', 'Data analysis software', 'Report generation tools']
  Reasoning:   This task requires gathering data from various sources, analyzing that data, and synthesizing it into a coherent report. An AI agent can efficiently handle data collection, perform comparative analysis, and generate reports, which would be time-consuming and complex for a human to do manually.

Task: Monitor stock prices and alert me when AAPL drops below $150
  Needs Agent: True
  

## 2.7 LLM Limitations — What the Brain Cannot Do

Understanding limitations is critical for building robust agents:

| Limitation | Impact on Agents | Mitigation |
|-----------|-----------------|------------|
| **Hallucination** | Agent may invent tool results | Stop before `Observation:`, inject real results |
| **Context window** | Can't remember infinite history | Summarization, memory management |
| **Math errors** | Unreliable calculations | Delegate math to calculator tools |
| **Knowledge cutoff** | No access to recent events | Provide tools for web search |
| **Inconsistency** | Same query → different actions | Low temperature, structured output |
| **Prompt injection** | Malicious inputs hijack behavior | Input validation, system prompt hardening |

The key insight: **agents compensate for LLM weaknesses by delegating to tools**.
- Bad at math? → Calculator tool
- No recent knowledge? → Search tool
- Can't access databases? → SQL tool
- Hallucinating? → Retrieval (RAG) tool

## 💡 Exercise 2: Structured Agent Decision-Making

**Task**: Create a Pydantic model for an agent that can route customer support tickets.

The model should output:
- `category`: one of ['billing', 'technical', 'general', 'urgent']
- `priority`: 1-5 integer
- `needs_human`: boolean
- `suggested_response`: draft response string
- `reasoning`: why this categorization was chosen

In [ ]:
# Exercise 2: Build a ticket routing model
# YOUR CODE HERE

from pydantic import BaseModel, Field
from typing import Literal

class TicketRouting(BaseModel):
    """Route a customer support ticket to the right team."""
    # Define your fields here
    pass

# Test with sample tickets
test_tickets = [
    'I was charged twice for my subscription this month!',
    'How do I reset my password?',
    'Your service has been down for 3 hours and we are losing money!'
]

# Use structured_llm to route each ticket
# routing_llm = llm.with_structured_output(TicketRouting)
# for ticket in test_tickets:
#     result = routing_llm.invoke(f'Route this support ticket: {ticket}')
#     print(result)


## 2.8 Preview — Από LLM σε Agent με μία γραμμή

Τώρα που είδαμε πώς το LLM είναι «brain in a jar», ας δούμε πόσο εύκολο είναι να του δώσουμε hands μέσω της `create_agent`. Στα επόμενα notebooks θα δούμε τι κάνει εσωτερικά.

In [ ]:
from langchain_core.tools import tool
from langchain_core.messages import HumanMessage
from langchain.agents import create_agent

@tool
def word_count(text: str) -> int:
    """Count words in a string."""
    return len(text.split())

preview_agent = create_agent(
    model=llm,
    tools=[word_count],
    prompt='Use the word_count tool when the user asks how many words a string has.',
)

r = preview_agent.invoke({'messages': [HumanMessage(content='How many words in: hello brave new agentic world?')]})
print(r['messages'][-1].content)


## 📝 Summary

| Concept | Key Takeaway |
|---------|-------------|
| **Autoregressive** | LLMs predict one token at a time — this is why they need stopping conditions |
| **Message Roles** | system (identity), user (input), assistant (output), tool (observations) |
| **Temperature** | Low (0-0.3) for agent reliability, higher for creativity |
| **Structured Output** | Pydantic models ensure reliable, parseable agent decisions |
| **LangChain LCEL** | `prompt | model | parser` — composable chain building |
| **Limitations** | Hallucination, context limits, math errors → mitigated by tools |

### What's Next

In **Notebook 03: Agent Loops and Core Architecture**, we will build our first complete agent loop — from scratch and with LangGraph's StateGraph.